# Workspaces

A {class}`~neurosetta.api.Workspace` captures the **scientific state of an analysis**
involving a {class}`~neurosetta.api.Forest`: the morphologies, named subsets,
workspace-level metadata, and explicitly registered results.

Save it as one compressed ``.nrw`` file, close Python, reopen later, and continue
from the reconstructed Forest / selections / results.

| Concept | Role |
|---------|------|
| `Tree` / `Forest` | Morphology objects |
| `.nr` | Persistence of an **individual** Tree |
| `Workspace` | Analysis **session** around one Forest |
| `.nrw` | Persistence of a Workspace |

A Workspace **owns** a Forest by composition — it is not a Forest subclass.
Workspace metadata stays on the Workspace; it is never broadcast onto Trees.

See {doc}`../api/workspace` and {doc}`../api/io` for the full API reference.


## Setup

Load the bundled example Forest. We'll use it as the primary morphology
collection for the Workspace.


In [1]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import neurosetta as nr

forest = nr.load_example_data()
forest.summary()


ID,nodes,branches,leaves,cable,units,isReduced,Flag
720575940596125868,2010,99,111,404.2,micron,False,False
720575940599459782,1408,90,99,298.0,micron,False,False
720575940599704006,2043,140,156,429.4,micron,False,False
720575940599729862,2299,149,166,494.6,micron,False,False
TOTAL,7760,478,532,1626.3,micron,,
MEAN,1940,119.5,133,406.6,micron,,


## Create a Workspace

Pass the Forest plus optional ``name``, ``description``, and JSON-compatible
``metadata``. Metadata here is **workspace-level** — separate from Tree
``metadata``.


In [2]:
ws = nr.Workspace(
    forest,
    name="Example FlyWire analysis",
    description="Morphometrics and subsets on the bundled example neurons",
    metadata={
        "dataset": "example",
        "coordinate_space": "FlyWire",
        "notes": "tutorial workspace",
    },
)
ws


Workspace(name='Example FlyWire analysis', trees=4, selections=0, results=0)

In [3]:
print("name:", ws.name)
print("description:", ws.description)
print("metadata:", ws.metadata)
print("primary forest ids:", ws.forest.ids())


name: Example FlyWire analysis
description: Morphometrics and subsets on the bundled example neurons
metadata: {'dataset': 'example', 'coordinate_space': 'FlyWire', 'notes': 'tutorial workspace'}
primary forest ids: [720575940596125868, 720575940599459782, 720575940599704006, 720575940599729862]


## Named selections

Selections are persistent named subsets of the primary Forest. They store Tree
**IDs** (references), not copies of Tree objects — so the archive does not
duplicate morphologies.

You can pass a sub-Forest, a list of IDs, or individual Trees. Missing IDs raise
``KeyError`` (nothing is silently dropped). Duplicate names raise unless you
pass ``overwrite=True``.


In [4]:
ids = forest.ids()

# sub-Forest
ws.add_selection("first_two", forest[:2])

# explicit IDs (order is preserved)
ws.add_selection("last_two", [ids[-2], ids[-1]])

# single tree
ws.add_selection("anchor", forest[0])

ws.list_selections()


['first_two', 'last_two', 'anchor']

In [5]:
subset = ws.get_selection("first_two")
print(subset)
print("ids:", subset.ids())
print("node counts:", subset.count_nodes())

# stored ID list (copy)
print("selection ids:", ws.selection_ids("last_two"))


Forest(n=2, ids=[720575940596125868, 720575940599459782])
ids: [720575940596125868, 720575940599459782]
node counts: [2010, 1408]
selection ids: [720575940599704006, 720575940599729862]


In [6]:
# overwrite an existing name
ws.add_selection("first_two", forest[:1], overwrite=True)
print("after overwrite:", ws.get_selection("first_two").ids())

# restore a useful subset for the rest of the tutorial
ws.add_selection("first_two", forest[:2], overwrite=True)


after overwrite: [720575940596125868]


## Analysis results

Results must be **registered explicitly** — NeuRosetta does not scrape your
namespace or pickle arbitrary objects.

Supported v1 types:

| Python type | On-disk id | Notes |
|-------------|------------|-------|
| JSON-compatible data | ``json`` | dict / list / scalars / nested |
| ``numpy.ndarray`` | ``numpy`` | non-object dtype; ``allow_pickle=False`` |
| ``pandas.DataFrame`` | ``dataframe`` | table-oriented JSON round-trip |

Unsupported types fail at save with a clear ``TypeError``.

### Tree-bound state vs workspace results

If something is already bound on a Tree (graph properties, metadata, mesh,
synapses) and ``.nr`` already persists it, **do not** duplicate it as a
workspace artifact. Use ``add_result`` for tables / arrays / configs that live
**outside** individual Trees.


In [7]:
# DataFrame from the descriptives registry
metrics = nr.describe(forest, domains="topology")
ws.add_result("metrics", metrics)
metrics.head()


,neuron_id,topology.count_roots,topology.count_nodes,topology.count_edges,topology.count_leaves,topology.count_branches,topology.count_transitive_nodes,topology.count_sections,topology.count_bifurcations,topology.count_core_nodes,...,topology.max_width,topology.mean_width,topology.median_width,topology.binary_ratio,topology.degrees.count,topology.degrees.mean,topology.degrees.std,topology.degrees.median,topology.degrees.q25,topology.degrees.q75
0,720575940596125868,1.0,2010.0,2009.0,111.0,99.0,1799.0,210.0,90.0,211.0,...,25.0,2.847025,1.0,0.942857,2010.0,1.999005,0.351258,2.0,2.0,2.0
1,720575940599459782,1.0,1408.0,1407.0,99.0,90.0,1219.0,188.0,81.0,189.0,...,19.0,3.108168,1.0,0.957447,1408.0,1.998580,0.388029,2.0,2.0,2.0
2,720575940599704006,1.0,2043.0,2042.0,156.0,140.0,1747.0,295.0,126.0,296.0,...,28.0,3.456853,1.0,0.949153,2043.0,1.999021,0.407947,2.0,2.0,2.0
3,720575940599729862,1.0,2299.0,2298.0,166.0,149.0,1983.0,315.0,135.0,316.0,...,31.0,3.916525,1.0,0.946032,2299.0,1.999130,0.400086,2.0,2.0,2.0


In [8]:
# JSON-compatible parameters / provenance
ws.add_result(
    "params",
    {
        "pipeline": "example-tutorial",
        "domains": ["topology"],
        "n_trees": len(forest),
    },
)

# NumPy array (e.g. pooled coordinates of the first tree)
coords = forest[0].get_node_coordinates()
ws.add_result("anchor_coords", coords)

print("results:", ws.list_results())
print("coords shape / dtype:", coords.shape, coords.dtype)


results: ['metrics', 'params', 'anchor_coords']
coords shape / dtype: (2010, 3) float64


## Save a ``.nrw`` archive

{func}`~neurosetta.save_workspace` writes a ZIP archive. Bare stems get
``.nrw`` appended. Writes are atomic (temp file + replace), so a failed save
will not clobber an existing valid archive.

Inside the archive:

```text
analysis.nrw
├── manifest.json          # versioned index
├── forest/t000000.nr …    # Trees via existing .nr I/O
└── artifacts/a000000.*    # registered results
```

Tree names / IDs are **not** used as archive filenames — opaque refs like
``t000000`` are. Forest order is defined by the manifest.


In [9]:
out_dir = Path(tempfile.mkdtemp(prefix="nrw-tutorial-"))
path = nr.save_workspace(ws, out_dir / "example_analysis")
print("wrote:", path)
print("exists:", path.is_file(), "size (bytes):", path.stat().st_size)

# live Forest is still usable after save
print("still have trees:", len(ws.forest), "ids:", ws.forest.ids()[:2], "...")


wrote: /tmp/nrw-tutorial-_2g9atqw/example_analysis.nrw
exists: True size (bytes): 306072
still have trees: 4 ids: [720575940596125868, 720575940599459782] ...


## Inspect without loading Trees

{func}`~neurosetta.inspect_workspace` reads ``manifest.json`` only — useful for
quickly seeing what is inside a large archive. Pass ``verify=True`` to check
SHA-256 digests without reconstructing morphologies.


In [10]:
info = nr.inspect_workspace(path)
info


{'name': 'Example FlyWire analysis',
 'description': 'Morphometrics and subsets on the bundled example neurons',
 'metadata': {'dataset': 'example',
  'coordinate_space': 'FlyWire',
  'notes': 'tutorial workspace'},
 'schema_version': 1,
 'neurosetta_version': '0.1.0',
 'created_at': '2026-10-02T14:17:23Z',
 'tree_count': 4,
 'selections': {'first_two': 2, 'last_two': 2, 'anchor': 1},
 'results': {'metrics': 'dataframe',
  'params': 'json',
  'anchor_coords': 'numpy'}}

In [11]:
print("name:", info["name"])
print("schema_version:", info["schema_version"])
print("neurosetta_version:", info["neurosetta_version"])
print("tree_count:", info["tree_count"])
print("selections:", info["selections"])
print("results:", info["results"])


name: Example FlyWire analysis
schema_version: 1
neurosetta_version: 0.1.0
tree_count: 4
selections: {'first_two': 2, 'last_two': 2, 'anchor': 1}
results: {'metrics': 'dataframe', 'params': 'json', 'anchor_coords': 'numpy'}


## Load and continue

{func}`~neurosetta.load_workspace` validates the format / schema, reconstructs
the Forest in manifest order, restores selections and results, and returns a
normal {class}`~neurosetta.api.Workspace`.

After load, Tree ``file_path`` metadata points at the **``.nrw`` archive**, not
a deleted temporary extract path.


In [12]:
restored = nr.load_workspace(path)
restored


Workspace(name='Example FlyWire analysis', trees=4, selections=3, results=3)

In [13]:
assert len(restored.forest) == len(forest)
assert restored.forest.ids() == forest.ids()
assert restored.name == ws.name
assert restored.metadata == ws.metadata

subset = restored.get_selection("first_two")
print("selection ids:", subset.ids())
print("selection node counts:", subset.count_nodes())

params = restored.get_result("params")
print("params:", params)

coords2 = restored.get_result("anchor_coords")
print("coords match:", np.allclose(coords2, coords))

metrics2 = restored.get_result("metrics")
print("metrics rows:", len(metrics2), "columns:", list(metrics2.columns)[:5], "...")


selection ids: [720575940596125868, 720575940599459782]
selection node counts: [2010, 1408]
params: {'pipeline': 'example-tutorial', 'domains': ['topology'], 'n_trees': 4}
coords match: True
metrics rows: 4 columns: ['neuron_id', 'topology.count_roots', 'topology.count_nodes', 'topology.count_edges', 'topology.count_leaves'] ...


In [14]:
# file_path points at the workspace archive
for tree in restored.forest:
    print(tree.ID, "→", tree.metadata["file_path"])
    break


720575940596125868 → /tmp/nrw-tutorial-_2g9atqw/example_analysis.nrw


## Morphology and Tree-bound state survive via ``.nr``

Each Tree inside the Workspace is stored with the normal ``.nr`` serializer, so
graph structure, names, IDs, units, and bound Tree state round-trip the same way
as {func}`~neurosetta.save` / {func}`~neurosetta.load`.


In [15]:
# compare a few morphology summaries before / after
orig = forest[0]
load = restored.forest.by_id(orig.ID)

print("ID / name:", load.ID, load.name)
print("nodes:", orig.count_nodes(), "→", load.count_nodes())
print("leaves:", orig.count_leaves(), "→", load.count_leaves())
print("cable:", f"{orig.get_total_cable_length():.2f}", "→", f"{load.get_total_cable_length():.2f}")
print("units:", load.metadata.get("units"))


ID / name: 720575940596125868 720575940596125868
nodes: 2010 → 2010
leaves: 111 → 111
cable: 404.24 → 404.24
units: micron


## What is intentionally not serialized

Workspaces persist **scientific state**, not Python runtime state:

- Qt widgets / GUI windows
- vedo actors / matplotlib figures
- executors, open handles, lambdas, callables
- regenerable caches
- arbitrary objects via pickle

Register what you care about with ``add_selection`` / ``add_result``. That keeps
persistence predictable and auditable.

Schema versioning is independent of the NeuRosetta package version. v1 loads
``schema_version == 1``; newer schemas raise
{class}`~neurosetta.io.WorkspaceVersionError`.

Next: {doc}`plotting` · {doc}`forests` · API {doc}`../api/workspace`
